In [10]:
print('test')

test


### 챗봇 예제(역할 부여)

In [11]:
# 사전 설치 : pip install gradio langchain-ollama
# import ollama : Ollama 팀이 직접 관리하는 공식 Python 라이브러리, 
# Ollama 직접 제어 시 사용
import ollama

In [12]:
# 챗봇의 기본적 질문, 답변 역할
def ask_gemma(question):
    # ollama를 사용하여 모델로부터 응답 생성
    chatbot_role = "너는 전문 심리 상담가야. 질문에 대한 답은 3줄 이내로 짧게 해줘."
    response = ollama.chat(model='gemma4:e2b', messages=[
        {"role": "system", "content": chatbot_role},    # 챗봇의 기본 역할 부여
        {"role": "user", "content": question},           # 질문
    ])
    
    return response['message']['content']

In [ ]:
question = "행복하기 위해 어떻게 하면 좋을까?"
response = ask_gemma(question)
print(response)

### 챗봇 예제(Gradio 사용)

In [ ]:
from langchain_ollama import ChatOllama
# HumanMessage: 사용자가 보낸 메시지, AIMessage : LLM의 메시지
from langchain_core.messages import HumanMessage, AIMessage
import gradio as gr

d:\pythonscript\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ChatOllama 모델 초기화
model = ChatOllama(model="gemma4:e2b", temperature=0.7, verbose=False)

In [ ]:
# 채팅 기록을 포함하여 응답을 생성하는 함수
def chat(message, history):
    # 이전 대화 기록을 ChatOllama 형식으로 변환
    chat_history = []
    for human, ai in history:
        chat_history.append(HumanMessage(content=human))
        chat_history.append(AIMessage(content=ai))
        
    # 현재 메시지 추가
    chat_history.append(HumanMessage(content=message))
    
    # 모델을 사용하여 응답 생성
    response = model.invoke(chat_history)
    
    return response.content

In [ ]:
# Gradio 인터페이스 설정
demo = gr.ChatInterface(
    fn=chat,
    examples=[
        "안녕하세요!",
        "인공지능에 대해 설명해주세요.",
        "파이썬의 장점은 무엇인가요?"
    ],
    title="AI 챗봇",
    description="질문을 입력하면 AI가 답변합니다."
)

In [ ]:
# 서버 실행
demo.launch(server_port=7861, server_name="0.0.0.0")

* Running on local URL:  http://0.0.0.0:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
demo.close()

Closing server running on port: 7861


### 챗봇 예제(Gradio + csv 사용)

In [ ]:
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
# 특정 문자(예: 줄바꿈, 공백)를 기준으로 텍스트를 분할하는 기능을 제공
# from langchain_text_splitters import CharacterTextSplitter   
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
import gradio as gr

In [ ]:
df = pd.read_csv("../dataset/indata_kor.csv", encoding='CP949')

In [ ]:
df.tail()

,inputs,response
27,한국폴리텍대학 스마트금융과의 최종 아웃풋은 어떤건가요?,스마트금융과는 찍어내기식의 포트폴리오가 아니라 매년 업체에서 요구하는 기술 및 주제...
28,한국폴리텍대학 스마트금융과의 최종 포트폴리오는 어떤건가요?,유튜브 채널에서 스마트금융과를 검색하시면 한국폴리텍대학 스마트금융과 포트폴리오 발표...
29,한국폴리텍대학 스마트금융과 면접시에는 어떤걸 준비하고 가면 될까요?,영문 타자연습 및 스마트금융과에 대한 열정을 보여주면 좋다
30,한국폴리텍대학 스마트금융과 입학 전까지 어떤걸 공부하면 될까요?,기본적인 OA를 잘 다루고 기본코드는 HKCODE의 기본 내용은 보고오면 됨. 파이...
31,한국폴리텍대학 스마트금융과는 대면/비대면 수업 어떻게 진행되나요?,대면으로 진행합니다.


In [ ]:
# 질문과 답변 CSV이므로
# "한 행 = 하나의 Q&A 문서"로 구성
texts = [
    f"질문: {row['inputs']}\n답변: {row['response']}"
    for _, row in df.iterrows()
]

In [ ]:
# 임베딩 모델 초기화
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/distiluse-base-multilingual-cased-v2")
# 모델 이름 : 조직이름(sentence-transformers) 다양한 작업 가능(all)-MS사 경령화 트랜스포머모델(MiniLM)-모델의 레이어수(L6)-모델이 버전(v2)
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

C:\Users\human-10\AppData\Local\Temp\ipykernel_8280\1471889035.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
d:\pythonscript\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\human-10\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, s

In [ ]:
# 벡터 데이터베이스 생성
# from_texts : 임베딩으로 변환된 벡터를 FAISS 인덱스에 저장
vectorstore = FAISS.from_texts(texts, embeddings)

In [ ]:
# ChatOllama 모델 초기화
llm = ChatOllama(model="gemma2", temperature=0.1)

In [ ]:
# 프롬프트 템플릿 정의 : 
# 모델이 제공된 Context(청크 검색 결과) 만을 기반으로 답변하도록 유도
prompt = ChatPromptTemplate.from_messages([
    ("system", "Yor are a helpful assistant. Answer based on the provided context."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}\n\nContext: {context}")
])

In [ ]:
# 문서 포맷팅 : RAG 시스템에서 검색된 문서들(docs)을 
# LLM이 이해하기 좋은 형태의 하나의 문자열로 변환

def format_docs(docs):
    if not docs:
        return "No context available"
    result = []     # 결과를 모아둘 빈 리스트 생성
    for doc in docs:
        # 객체(doc)가 특정 속성(page_content)을 가지고 있는지 확인
        if hasattr(doc, 'page_content'):
            result.append(doc.page_content)
    return "\n".join(result)

In [ ]:
# 리트리버 설정
retriver = vectorstore.as_retriever(search_kwargs={"k": 1})

In [ ]:
# 채팅 기록을 포함하여 응답을 생성하는 함수
def chat(message, history):
    chat_history = []
    
    # history 순회 처리
    for item in history:
        # 1. 최신 메시지 딕셔너리 구조인 경우: 
        # 예시 ==> [{"role": "user", "content": "안녕"}, 
        #          {"role": "assistant", "content": "반가워요"}]
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content", "")
            if role == "user":
                chat_history.append(HumanMessage(content=content))
            elif role == "assistant":
                chat_history.append(AIMessage(content=content))
                
        # 2. 리스트/튜플 형태인 경우: ['질문', '답변']
        elif isinstance(item, (list, tuple)):
            if len(item) == 2:
                chat_history.append(HumanMessage(content=itme[0]))
                chat_history.append(AIMessage(content=item[1]))
            elif len(item) == 1:
                chat_history.append(HumanMessage(content=item[0]))
        
    # 현재 메시지 추가
    chat_history.append(HumanMessage(content=message))
        
    # 모델 호출
    response = model.invoke(chat_history)
    return response.content

In [ ]:
# Gradio 인터페이스 설정
demo = gr.ChatInterface(
    fn=chat,
    examples=[
        "한국폴리텍대학 스마트금융과 입학 전까지 어떤걸 공부하면 될까요?",
        "스마트금융과에 대해 설명해주세요",
        "한국폴리텍대한 추천할만한 학과 하나를 소개해주세요."    ]
)
title="대학 정보 AI 챗봇",
description = "스마트금융과에 대한 질문을 입력하면 AI가 CSV데이터를 참고하여 한글로 답변합니다."

In [ ]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
demo.close()

Closing server running on port: 7860


### 챗봇 예제(STT: 음성을 텍스트로 전환)

In [ ]:
# 사전 설치 : pip install openai-whisper
# ffmpeg 사전 설치 및 환경변수 path 설정(경로/bin)
# 코랩 audio에서 테스트할 샘플 오디오 파일 다운
import os
from dotenv import load_dotenv  # 환경변수 로드가 필요한 경우
import whisper
import gradio as gr

d:\pythonscript\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ffmpeg 경로 명시적 설정
# os.environ["FFMPEG_BINARY"] = "C:/aiproject/ffmpeg/bin/ffmpeg.exe"
os.environ["PATH"] += os.pathsep + r"C:\ffmpeg\bin"
os.environ["FFMPEG_BINARY"] = r"C:\ffmpeg\bin\ffmpeg.exe"

In [ ]:
def transcribe_audio(audio_path):
    # Whisper 모델 로드
    # tiny, base(Vram:~1GB). small(Vram:~2GB). medium(Vram:~5GB), large(Vram:~10GB)
    model = whisper.load_model("base")
    
    # 오디오 파일 전사
    result = model.transcribe(audio_path)
    
    # 전사된 텍스트 반환
    return result["text"]
    

In [ ]:
def process_audio(audio):
    if audio is None:
        return "오디오 파일을 업로드해주세요"
    try:
        transcribed_text = transcribe_audio(audio)
        return transcribed_text
    except Exception as e:
        return f"오류가 발행샣ㅆ습니다: {str(e)}"

In [ ]:
# Gradio 인터페이스 생성
iface = gr.Interface(
    fn=process_audio,
    inputs=gr.Audio(type="filepath", label="MP3 파일 업로드"),
    outputs=gr.Textbox(label="음성 분석 결과", lines=10),
    title="MP3 to text Converter",
    description="MP3 파일을 업로드하면 텍스트로 변환합니다."
)

In [ ]:
# 디버그 모드로 Gradio 인터페이스 실행
iface.launch(server_port=7861, server_name="0.0.0.0", debug=True)

* Running on local URL:  http://0.0.0.0:7861
* To create a public link, set `share=True` in `launch()`.


Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "C:\python312\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] 현재 연결은 원격 호스트에 의해 강제로 끊겼습니다
100%|███████████████████████████████████████| 139M/139M [00:24<00:00, 5.95MiB/s]
d:\pythonscript\.venv\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Keyboard interruption in main thread... closing server.


In [ ]:
iface.close()

### 이미지 분류 예제(Gradio + gemma2 사용)

In [2]:
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image
import requests
from io import BytesIO
from langchain_ollama import ChatOllama

d:\pythonscript\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# TensorFlow MobileNetV2 모델 로드
# 사전 훈련된 가중치를 사용
model = tf.keras.applications.MobileNetV2(weights="imagenet")

14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [5]:
OLLAMA_SERVER = "http://localhost:11434"
MODEL_NAME = "gemma2"

In [6]:
# Ollama를 사용해 음식 설명 생성
def get_food_description_with_langchain(food_name):
    """
    LangChain ChatOllama를 사용하여 음식 설명 생성
    """
    try:
        chat = ChatOllama(base_url=OLLAMA_SERVER, model=MODEL_NAME)
        prompt = f"{food_name}에 대해 특징, 효능, 요리 레시피 설명해줘. 설명은 한국어로 해줘"
        response = chat.invoke(prompt)
        return response.content
    except Exception as e:
        return f"Failed to retrieve description: {e}"

In [ ]:
# 이미지 예측 함수
def predict_image_with_description(image_url):
    """
    이미지 URL을 받아 음식 예측과 Ollama 설명을 반환
    """
    try:
        # URL에서 이미지 가져오기
        response = requests.get(image_url)
        # BytesIO 사용하여 이미지 열기
        image = Image.open(BytesIO(response.content)).resize((224, 224))
        
        # 이미지를 숫자 배열로 변환
        image_array = tf.keras.preprocessing.image.img_to_array(image)
        # 모델이 한 번에 여러 이미지를 처리할 수 있게 "배치"라는 차원을 추가
        image_array = tf.expand_dims(image_array, axis=0)
        # 이미지 픽셀 값을 모델이 학습할 때 사용했던 범위로 조정 전처리
        image_array = tf.keras.applications.mobilenet_v2.preprocess_input(image_array)
        
        # 예측 수행(Top=3)
        predictions = model.predict(image_array)
        decoded_predictions = tf.keras.applications.mobilenet_v2.decode_predictions(3)[0]
        
        # 예측 결과 형식화
        # 예측 결과를 Gradio의 Label 컴포넌트가 요구하는 형식으로 변환
        result = {label: float(prob) for (_, label, prob) in decoded_predictions}
        
        # 가장 높은 확률의 예측값으로 Ollama 설명 생성
        # 가장 확률이 높은 음식 이름
        top_food = decoded_predictions[0][1]
        description = get_food_description_with_langchain(top_food)
        
        return result, description   # 예측 결과와 Ollama 설명 반환
    
    except Exception as e:
        return {"error": 1.0}, f"Error: {e}"    # 에러 발생 시 기본값 반환

In [18]:
# Gradio 인터페이스 생성
iface = gr.Interface(
    fn=predict_image_with_description,
    inputs=gr.Textbox(label="이미지 URL 입력"),
    outputs=[
        # 상위 3개 예측 결과
        gr.Label(num_top_classes=3, label="예측 결과"),  
        # Ollama로 생성한 설명 출력
        gr.Textbox(label="음식 설명", interactive=False)    
    ],
    title="음식 이미지 분류 및 설명 생성기",
    description="이미지 URL을 입력하면 음식 분류 결과와 설명을 제공합니다."
)

In [19]:
iface.launch(server_port=7861, server_name="0.0.0.0", debug=True)

* Running on local URL:  http://0.0.0.0:7861
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


In [21]:
iface.close()

### 자기소개서 도우미 챗봇 예제(Gradio)

In [22]:
import os
import gradio as gr
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
# LCEL과 함께 사용할 출력 파서
from langchain_core.output_parsers import StrOutputParser
from fpdf import FPDF

In [23]:
# Ollama 설정(Gemma2 모델 사용)
os.environ["OLLAMA_API_BASE"] = "http://localhost:11434"
ollama_model = Ollama(model="gemma2")

C:\Users\human-10\AppData\Local\Temp\ipykernel_13732\1170127201.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  ollama_model = Ollama(model="gemma2")


In [24]:
# 다양한 템플릿 설정
TEMPLATES = {
    "취업": "다음 키워드와 예시를 바탕으로, 취업 지원을 위한 자기소개를 작성하세요.",
    "대학원": "제공된 키워드를 사용하여, 대학원 지원을 위한 자기소개서 초안 작성하세요.",
    "봉사활동": "주어진 키워드를 활용하여, 봉사활동 경험과 동기를 강조하는 자기소개서를 작성하세요."
}

In [25]:
# 언어지원 : 한국어, 영어, 일본어
LANGUAGES = {
    "한국어": "Please write the response in Korean",
    "영어": "Please write the response in English",
    "일본어": "Please write the response in Japanese"
    
}

In [26]:
# 자동 키워드 추천 함수
def recommend_keywords(purpose):
    if purpose == "취업":
        return "책임감, 팀워크, 문제 해결 능력"
    elif purpose == "대학원":
        return "연구 열정, 창의력, 학업 성취도"
    elif purpose == "봉사활동":
        return "사회적 책임감, 희생정신, 리더십"
    else:
        return ""
    

In [27]:
# 자기소개서 작성 함수
def generate_statement(purpose, language, keywords, example_sentence=None):
    if purpose not in TEMPLATES:
        return "지원 목적을 올바르게 선택해주세요."
    if language not in LANGUAGES:
        return "언어를 올바르게 선택해주세요."
    
    # 템플릿 생성
    template = TEMPLATES[purpose] + "\n\nKeywords: {keywords}\n" + LANGUAGES[language]
    if example_sentence:
        template += f"\n\nExample sentence: {example_sentence}"
        
    prompt = PromptTemplate(input_variables=["keywords"], template=template)
    
    # ======  LCEL 체인 사용 ======
    # 프롬프트, LLM, 출력 파서를 파이프(|)로 간단하게 연결합니다.
    # StrOutputParser는 LLM의 출력에서 문자열만 깔끔하게 추출해줍니다.
    chain = prompt | ollama_model | StrOutputParser()
    
    # .invoke()에 입력 변수를 딕셔너리 형태로 전달합니다.
    response = chain.invoke({"Keywords":keywords})
    
    return response

In [28]:
# PDF 저장 함수
def save_to_pdf(statement, filename="personal_statement.pdf"):
    pdf = FPDF()
    # pdf 문서에 새로운 페이지 추가
    pdf.add_page()
    # '맑은 고딕' 폰트 경로 설정
    pdf.add_font('MalgunGothic', '', r'C\Windows\Fonts\malgun.ttf', uni=True)
    # 폰트 설정
    pdf.set_font('MalgunGothic', size=12)
    # Mac용 폰트 경로 설정
    # font_path = "/System/Library/Fonts/AppleSDGothicNeo.ttc"
    # pdf.add_font('AppleSDGothic', '', font_path, uni=True)
    # pdf.set_font('AppleSDGothic', size=12)
    
    # PDF 문서에 텍스트를 추가, 셀의너비(0), 셀의 높이(10)
    pdf.multi_cell(0, 10, statement)
    # PDF 문서를 파일로 저장
    pdf.output(filename)
    return f"PDF 저장 완료: {filename}"

In [29]:
# Gradio 인터페이스
def chatbot_interface(purpose, language, keywords, example_sentence=None, save_pdf=False):
    statement = generate_statement(purpose, language, keywords, example_sentence)
    if save_pdf:
        save_to_pdf(statement)
    return statement

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# 다목적 자기소개서 작성 도우미")
    gr.Markdown("키워드와 추천 문장을 활용하여 취업, 대학원, 봉사활동 자기소개서를 생성하고 PDF로 저장하세요!")
    
    # 입력 영역
    with gr.Row():
        purpose_input = gr.Dropdown(label="지원 목적", choices=["취업", "대학원", "봉사활동"], value="취업")
        language_input = gr.Dropdown(label="언어 선택", choices=["한국어", "영어", "일본어"], value="한국어")
    
    recommend_keywords = gr.Textbox(label="추천 키워드", interactive=False)
    recommend_btn = gr.Button("키워드 추천")
    recommend_btn.click(recommend_keywords, inputs=[purpose_input], outputs=[recommend_keywords])
    
    with gr.Row():
        keywords_input = gr.Textbox(label="사용자 키워드 입력", placeholder="예: 책임감, 팀워크, 문제 해결 능력")
        example_sentence_input = gr.Textbox(
            
        )